# AURA — Train World Model

Train the cRSSM world model on Colab GPU (A100/T4).
Requires data generated by `01_data_generation.ipynb`.

In [ ]:
# Setup (Colab)
import os

if 'COLAB_GPU' in os.environ:
    !git clone https://github.com/SotoAlt/aura.git
    %cd aura
    !pip install -q -r world_model/requirements-colab.txt

    from google.colab import drive
    drive.mount('/content/drive')

    # Copy data from Drive to local SSD for faster I/O
    DRIVE_DATA = '/content/drive/MyDrive/aura/data/train'
    LOCAL_DATA = '/content/aura_data'
    !mkdir -p {LOCAL_DATA}
    !cp -r {DRIVE_DATA}/*.npz {LOCAL_DATA}/
    print(f'Copied data to local SSD: {LOCAL_DATA}')

    CHECKPOINT_DIR = '/content/drive/MyDrive/aura/checkpoints'
else:
    LOCAL_DATA = 'data/train'
    CHECKPOINT_DIR = 'checkpoints'

In [ ]:
# Verify JAX sees GPU
import jax
print(f'JAX devices: {jax.devices()}')
assert any('gpu' in str(d).lower() or 'cuda' in str(d).lower()
           for d in jax.devices()) or True, 'No GPU found (continuing on CPU)'

In [ ]:
# Verify data
from pathlib import Path
data_path = Path(LOCAL_DATA)
episodes = sorted(data_path.glob('episode_*.npz'))
print(f'Training episodes: {len(episodes)}')
assert len(episodes) > 0, f'No episodes found in {LOCAL_DATA}'

## Train

Use `aura` config for GPU training, `aura_debug` for quick CPU testing.

In [ ]:
CONFIG = 'aura'  # 'aura' for GPU, 'aura_debug' for CPU test
STEPS = 100000   # Reduce for testing
CHECKPOINT = f'{CHECKPOINT_DIR}/aura-v0.1.ckpt'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

from world_model.train import train

params, opt_state = train(
    cfg_name=CONFIG,
    data_dir=LOCAL_DATA,
    steps=STEPS,
    checkpoint_path=CHECKPOINT,
    log_every=100,
    save_every=5000,
    val_every=1000,
    wandb_enabled=True,
)

In [ ]:
# Verify checkpoint was saved
ckpt_path = Path(CHECKPOINT)
if ckpt_path.exists():
    size_mb = ckpt_path.stat().st_size / 1e6
    print(f'Checkpoint: {CHECKPOINT} ({size_mb:.1f} MB)')
else:
    print('WARNING: Checkpoint not found!')

## Quick Visualization

In [ ]:
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from world_model.dreamer.agent import make_rssm_config, load_config, imagine_trajectory
from world_model.dreamer.rssm import initial_state

cfg = load_config(CONFIG)
rssm_cfg = make_rssm_config(cfg)
init_s = initial_state(rssm_cfg, 1)

# Imagine with high energy
T = 20
actions = jax.nn.one_hot(jnp.zeros((1, T), dtype=jnp.int32), cfg['action_dim'])
ctx_high = jnp.ones((1, T, 16)) * 0.9
ctx_low = jnp.ones((1, T, 16)) * 0.1

rng = jax.random.key(0)
frames_high = np.array(imagine_trajectory(params, rssm_cfg, init_s, actions, ctx_high, rng))[0]
frames_low = np.array(imagine_trajectory(params, rssm_cfg, init_s, actions, ctx_low, rng))[0]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(np.clip(frames_high[i * 4], 0, 1))
    ax.set_title(f'High t={i*4}')
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(np.clip(frames_low[i * 4], 0, 1))
    ax.set_title(f'Low t={i*4}')
    ax.axis('off')
plt.suptitle('Imagined Frames: High vs Low Energy Audio')
plt.tight_layout()
plt.show()